In [1]:
# Cell 1: Confirm Colab gave us a GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [2]:
# Cell 2: Install zstd (unpacking) and pciutils + lshw (GPU detection)
!apt-get install -y -q zstd pciutils lshw

Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  libpci3 pci.ids usb.ids
The following NEW packages will be installed:
  libpci3 lshw pci.ids pciutils usb.ids zstd
0 upgraded, 6 newly installed, 0 to remove and 33 not upgraded.
Need to get 1,583 kB of archives.
After this operation, 5,343 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 pci.ids all 0.0~2024.03.31-1ubuntu0.1 [275 kB]
Get:2 http://archive.ubuntu.com/ubuntu noble/main amd64 libpci3 amd64 1:3.10.0-2build1 [36.5 kB]
Get:3 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 lshw amd64 02.19.git.2021.06.19.996aaad9c7-2ubuntu0.24.04.1 [334 kB]
Get:4 http://archive.ubuntu.com/ubuntu noble/main amd64 pciutils amd64 1:3.10.0-2build1 [69.7 kB]
Get:5 http://archive.ubuntu.com/ubuntu noble/main amd64 usb.ids all 2024.03.18-1 [223 kB]
Get:6 http://archive.ubuntu.com/ubuntu noble-update

In [3]:
# Cell 3: Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [4]:
# Cell 4: Start the Ollama server in the background
!nohup ollama serve > ollama.log 2>&1 &
!sleep 8

In [5]:
# Cell 5: Install the Ollama Python client
!pip install -q ollama

In [6]:
# Cell: Download the model through the Python client
import ollama

print(ollama.pull("gemma4:e2b").status)

success


In [7]:
# Cell 7: Check the server and the model
!ollama list

NAME          ID              SIZE      MODIFIED               
gemma4:e2b    b37049369adf    4.6 GB    Less than a second ago    


In [8]:
%%writefile tools.py
"""Tools for interacting with the product catalog (synthetic data)."""

PRICES = {
    "mccb": 145.00,
    "acb": 2850.00,
    "busbar": 95.00,
    "current transformer": 38.00,
    "cable gland": 25.00,
}


def lookup_price(part_name: str) -> str:
    """Return the unit price of one part, as text (for example 95.0).
    Valid part names: mccb (moulded case circuit breaker),
    acb (air circuit breaker), busbar (copper busbar, per metre),
    current transformer, cable gland.
    Use the exact name in lowercase.
    If the name is not valid, the reply says so and lists the valid part names.
    """
    #Normalize the part name before look up
    name = part_name.strip().lower()
    if name in PRICES:
        return str(PRICES[name])
    return f"unknown part '{part_name}'. Valid part: {', '.join(PRICES)}"


def list_parts() -> str:
    """List every part in the catalog, separated by commas."""
    return ", ".join(PRICES)

def line_total(unit_price: float, quantity: int) -> float:
    """Return the total price for one quote line, rounded to 2 decimal places.
    unit_price is the price of one item in pounds. Use the value from lookup_price.
    quantity is the number of items, as a whole number.
    """
    # Round to 2 decimals so the amount is a clean currency value
    return round(unit_price * quantity, 2)

Writing tools.py


In [9]:
# Cell 9: Check that tools.py works on Colab
import tools

print(tools.lookup_price("busbar"))
print(tools.list_parts())
print(tools.line_total(95.0, 3))

95.0
mccb, acb, busbar, current transformer, cable gland
285.0


In [10]:
%%writefile agent.py

"""Runs the tools from tools.py safely, so a bad request from the model never crashes the agent."""

import ollama
MODEL = "gemma4:e2b"
from tools import line_total, list_parts, lookup_price

TOOLS = {
    "lookup_price": lookup_price,
    "list_parts": list_parts,
    "line_total": line_total,
}


def run_tool(name: str, arguments: dict) -> str:
    """Run one tool by name and return its result as text.
    An unknown tool name or wrong inputs return a message instead of raising an error."""
    # Reject names the model made up
    if name not in TOOLS:
        return f"Unknown tool '{name}'. Valid tools: {', '.join(TOOLS)}"
    try:
        return str(TOOLS[name](**arguments))
    except TypeError as error:
        # Raised when the inputs do not match the tool's parameters
        return f"Bad inputs for '{name}': {error}"


def run_agent(question:str, max_rounds:int = 5) -> str:
    """Ask the model, stop when it answers in text, and give up at the round limit."""

    # The history starts with the user's question
    messages = [{"role":"user", "content":question}]

    for round_number in range(1, max_rounds+1):
        print(f"Round {round_number}")

        # Ask the model, sending the whole history and the tool functions
        response = ollama.chat(
            model=MODEL,
            messages=messages,
            tools=list(TOOLS.values()),
            think=False,
        )

        # Keep the model's reply in the history
        messages.append(response.message)

        # No tool request means the model has finished
        if not response.message.tool_calls:
            return response.message.content
    return "Stopped: reached the round limit."
 


Writing agent.py


In [11]:
# Cell 11: Check that agent.py imports and its tool runner works
import agent

print(agent.run_tool("lookup_price", {"part_name": "busbar"}))
print(agent.run_tool("make_coffee", {}))
print(agent.run_agent("hello", max_rounds=0))

95.0
Unknown tool 'make_coffee'. Valid tools: lookup_price, list_parts, line_total
Stopped: reached the round limit.


In [12]:
# Cell 12: Run the agent on a question that needs no tool
from agent import run_agent

print(run_agent("What is the capital of France?"))

Round 1
The capital of France is Paris.


In [13]:
# Cell13: Reload agent.py so the new version is used
import importlib

import agent

importlib.reload(agent)
print("reloaded")

reloaded


In [14]:
# Cell 14: Run the agent on a question that needs no tool
print(agent.run_agent("What is the capital of France?"))

Round 1
The capital of France is Paris.


In [15]:
# Cell 15: Run the agent on a question that needs one tool
print(agent.run_agent("What is the price of one busbar?"))

Round 1
Round 2

